# QM9 Regression with GNNVisualizer

This notebook trains graph-level GCN, GAT, GraphSAGE, and GIN models on the PyTorch Geometric `QM9` dataset, then renders the four models with `GNNVisualizer`.

QM9 contains small molecules and 19 graph-level regression targets. The default target below is target index `0`, the dipole moment `mu`.

Source docs: [PyTorch Geometric QM9](https://pytorch-geometric.readthedocs.io/en/latest/generated/torch_geometric.datasets.QM9.html).

If imports fail in a fresh kernel, install the runtime packages first:

```bash
python3 -m pip install torch torch-geometric
```

Set `QM9_TARGET_INDEX` and `QM9_MAX_GRAPHS` before running the notebook to change the target property or sample size.

In [ ]:
import os
import sys
from pathlib import Path

repo_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import Markdown, display
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATConv, GCNConv, GINConv, SAGEConv, global_mean_pool

from gnn_exp import GNNVisualizer

from torch_geometric.datasets import QM9


In [ ]:
SEED = 7
torch.manual_seed(SEED)

TARGETS = [
    "mu", "alpha", "homo", "lumo", "gap", "r2", "zpve", "u0", "u",
    "h", "g", "cv", "u0_atom", "u_atom", "h_atom", "g_atom", "a", "b", "c",
]
TARGET_INDEX = int(os.environ.get("QM9_TARGET_INDEX", "0"))
MAX_GRAPHS = int(os.environ.get("QM9_MAX_GRAPHS", "512"))
BATCH_SIZE = 32
EPOCHS = 3
HIDDEN_CHANNELS = 16


def prepare_graph(data):
    data = data.clone()
    data.x = data.x.float()
    data.y = data.y.view(1, -1)[:, TARGET_INDEX].view(1)
    return data


dataset = QM9(root=str(repo_root / "data" / "qm9"))
generator = torch.Generator().manual_seed(SEED)
sample_count = min(MAX_GRAPHS, len(dataset))
indices = torch.randperm(len(dataset), generator=generator)[:sample_count].tolist()
sampled_graphs = [prepare_graph(dataset[int(index)]) for index in indices]
train_size = max(1, int(0.8 * len(sampled_graphs)))
train_graphs = sampled_graphs[:train_size]
valid_graphs = sampled_graphs[train_size:] or sampled_graphs[:1]
train_loader = DataLoader(train_graphs, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_graphs, batch_size=BATCH_SIZE, shuffle=False)
visual_data = sampled_graphs[0]
QUERY_PAIR = [0, min(1, visual_data.num_nodes - 1)]
NUM_FEATURES = visual_data.num_features
OUT_CHANNELS = 1

target_values = torch.stack([graph.y.view(()) for graph in train_graphs])
target_mean = target_values.mean()
target_std = target_values.std().clamp_min(1e-6)

display(Markdown(
    f"QM9 loaded with **{len(dataset):,} molecules**. "
    f"This demo samples **{len(sampled_graphs):,} molecules**, trains on **{len(train_graphs):,}**, "
    f"and predicts target `{TARGETS[TARGET_INDEX]}`. "
    f"The visualized molecule has **{visual_data.num_nodes} atoms**, "
    f"**{visual_data.edge_index.size(1)} directed bonds**, and **{NUM_FEATURES} node features**."
))


In [ ]:
class GCNGraphModel(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.act1 = nn.Tanh()
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.act2 = nn.Tanh()
        self.classifier = nn.Linear(hidden_channels, out_channels)

    def forward(self, x, edge_index, batch=None):
        x = x.float()
        if batch is None:
            batch = torch.zeros(x.size(0), dtype=torch.long, device=x.device)
        h = self.act1(self.conv1(x, edge_index))
        h = self.act2(self.conv2(h, edge_index))
        graph_embedding = global_mean_pool(h, batch)
        return self.classifier(graph_embedding)


class GATGraphModel(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        if hidden_channels % 2 != 0:
            raise ValueError("hidden_channels must be divisible by 2")
        heads = 2
        per_head_channels = hidden_channels // heads
        self.conv1 = GATConv(in_channels, per_head_channels, heads=heads, concat=True)
        self.act1 = nn.Tanh()
        self.conv2 = GATConv(hidden_channels, hidden_channels, heads=1, concat=False)
        self.act2 = nn.Tanh()
        self.classifier = nn.Linear(hidden_channels, out_channels)

    def forward(self, x, edge_index, batch=None):
        x = x.float()
        if batch is None:
            batch = torch.zeros(x.size(0), dtype=torch.long, device=x.device)
        h = self.act1(self.conv1(x, edge_index))
        h = self.act2(self.conv2(h, edge_index))
        graph_embedding = global_mean_pool(h, batch)
        return self.classifier(graph_embedding)


class GraphSAGEGraphModel(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.act1 = nn.Tanh()
        self.conv2 = SAGEConv(hidden_channels, hidden_channels)
        self.act2 = nn.Tanh()
        self.classifier = nn.Linear(hidden_channels, out_channels)

    def forward(self, x, edge_index, batch=None):
        x = x.float()
        if batch is None:
            batch = torch.zeros(x.size(0), dtype=torch.long, device=x.device)
        h = self.act1(self.conv1(x, edge_index))
        h = self.act2(self.conv2(h, edge_index))
        graph_embedding = global_mean_pool(h, batch)
        return self.classifier(graph_embedding)


class GINGraphModel(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GINConv(nn.Sequential(
            nn.Linear(in_channels, hidden_channels),
            nn.Tanh(),
            nn.Linear(hidden_channels, hidden_channels),
        ))
        self.act1 = nn.Tanh()
        self.conv2 = GINConv(nn.Sequential(
            nn.Linear(hidden_channels, hidden_channels),
            nn.Tanh(),
            nn.Linear(hidden_channels, hidden_channels),
        ))
        self.act2 = nn.Tanh()
        self.classifier = nn.Linear(hidden_channels, out_channels)

    def forward(self, x, edge_index, batch=None):
        x = x.float()
        if batch is None:
            batch = torch.zeros(x.size(0), dtype=torch.long, device=x.device)
        h = self.act1(self.conv1(x, edge_index))
        h = self.act2(self.conv2(h, edge_index))
        graph_embedding = global_mean_pool(h, batch)
        return self.classifier(graph_embedding)


In [ ]:
def scaled_target(batch):
    return (batch.y.view(-1, 1).float() - target_mean) / target_std


def evaluate_model(model, loader):
    model.eval()
    absolute_errors = []
    with torch.no_grad():
        for batch in loader:
            scaled_pred = model(batch.x, batch.edge_index, batch.batch)
            pred = scaled_pred * target_std + target_mean
            absolute_errors.append((pred.view(-1) - batch.y.view(-1).float()).abs())
    return float(torch.cat(absolute_errors).mean())


def train_model(model, loader, epochs=EPOCHS):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=1e-5)
    losses = []
    for _ in range(epochs):
        model.train()
        for batch in loader:
            optimizer.zero_grad()
            pred = model(batch.x, batch.edge_index, batch.batch)
            loss = F.mse_loss(pred, scaled_target(batch))
            loss.backward()
            optimizer.step()
            losses.append(float(loss.detach()))
    return {"final_scaled_mse": losses[-1], "valid_mae": evaluate_model(model, valid_loader)}


model_builders = {
    "GCN": lambda: GCNGraphModel(NUM_FEATURES, HIDDEN_CHANNELS, OUT_CHANNELS),
    "GAT": lambda: GATGraphModel(NUM_FEATURES, HIDDEN_CHANNELS, OUT_CHANNELS),
    "GraphSAGE": lambda: GraphSAGEGraphModel(NUM_FEATURES, HIDDEN_CHANNELS, OUT_CHANNELS),
    "GIN": lambda: GINGraphModel(NUM_FEATURES, HIDDEN_CHANNELS, OUT_CHANNELS),
}

models = {}
metrics_by_model = {}
for name, build_model in model_builders.items():
    torch.manual_seed(SEED)
    model = build_model()
    metrics_by_model[name] = train_model(model, train_loader)
    models[name] = model.eval()

rows = ["| Model | Final scaled MSE | Validation MAE |", "|---|---:|---:|"]
for name, metrics in metrics_by_model.items():
    rows.append(f"| {name} | {metrics['final_scaled_mse']:.4f} | {metrics['valid_mae']:.4f} |")
display(Markdown("\n".join(rows)))


The next cell creates one `GNNVisualizer` per trained model. The graph-level readout is captured from `global_mean_pool`.

In [ ]:
EXPECTED_LAYER_TYPES = {
    "GCN": "GCNConv",
    "GAT": "GATConv",
    "GraphSAGE": "SAGEConv",
    "GIN": "GINConv",
}


def make_visualizer(model, graph_data, query_pair):
    visualizer = GNNVisualizer(renderer="svg")
    visualizer.add_model(
        data=graph_data,
        model=model.eval(),
        subgraphSample=False,
        queries=[query_pair],
        mode="graph",
    )
    return visualizer


visualizers = {
    name: make_visualizer(model, visual_data, QUERY_PAIR)
    for name, model in models.items()
}

summary_rows = [
    "| Model | First layer | Aggregation | Graph pooling | Hidden width | Visualized nodes | Query |",
    "|---|---:|---:|---:|---:|---:|---:|",
]

for name, visualizer in visualizers.items():
    first_layer = visualizer.modelInfo["conv1"]
    assert first_layer["type"] == EXPECTED_LAYER_TYPES[name]
    assert len(visualizer.graphData["x"]) == visual_data.num_nodes
    assert "graphAggregation" in visualizer.intmData
    assert len(visualizer.intmData["act1"][0]) == HIDDEN_CHANNELS
    summary_rows.append(
        f"| {name} | `{first_layer['type']}` | `{first_layer.get('aggregation')}` | "
        f"`{visualizer.intmData['graphAggregation']['type']}` | "
        f"{len(visualizer.intmData['act1'][0])} | {len(visualizer.graphData['x'])} | "
        f"`{visualizer.queries}` |"
    )

display(Markdown("\n".join(summary_rows)))


## GCN

In [ ]:
display(visualizers["GCN"])

## GAT

In [ ]:
display(visualizers["GAT"])

## GraphSAGE

In [ ]:
display(visualizers["GraphSAGE"])

## GIN

In [ ]:
display(visualizers["GIN"])